# Notebook 4 - Modelo Único

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

import optuna
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

import funcoes_modelagem

C:\Users\Victor Dogo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pasta_dados = Path('dados')
arquivos_bases = sorted(pasta_dados.glob('3-*.pkl'))

bases = {}
for caminho_base in arquivos_bases:
    nome_base = caminho_base.stem.removeprefix('3-')
    bases[nome_base] = pd.read_pickle(caminho_base)
    globals()[nome_base] = bases[nome_base]

print('Bases carregadas:')
for nome_base, base in bases.items():
    print(f'{nome_base}: {base.shape[0]} linhas x {base.shape[1]} colunas')

Bases carregadas:
base_oot: 131 linhas x 31 colunas
base_teste: 188 linhas x 32 colunas
base_treino: 439 linhas x 32 colunas


Faremos algumas definições importantes que serão utilizadas para modelagem: o target e a variável de data a ser utilizada.

In [3]:
target = 'regularizou_30d'
datas = 'safra'

Durante os notebooks anteriores, definimos diversas variáveis, e abaixo iremos criar uma lista com todas elas para facilitar nosso trabalho.

In [4]:
colunas_excluidas = {
    target,
    datas,
    'score_cobranca_atualizado_30d',
    'flag_treino',
}

features = [
    coluna
    for coluna in base_treino.columns
    if coluna not in colunas_excluidas
    and not pd.api.types.is_string_dtype(base_treino[coluna])
]

print(f'{len(features)} features selecionadas:')
print(features)

20 features selecionadas:
['tempo_relacionamento_meses', 'faturamento_mensal_estimado', 'saldo_devedor', 'dias_atraso', 'qtd_contratos_ativos', 'qtd_parcelas_vencidas', 'limite_credito', 'utilizacao_limite_pct', 'qtd_contatos_ult_30d', 'promessa_pagamento_ult_30d', 'renegociacoes_ult_12m', 'score_regularizacao_legado', 'qtd_interacoes_ate_ref', 'dias_desde_ultima_interacao', 'qtd_canais_distintos_ate_ref', 'setor_cat', 'porte_cat', 'canal_preferencial_cat', 'risco_setorial_cat', 'regiao']


### Seleção inicial de features
Com toda a lista de features acima, faremos uma seleção inicial delas para identificar features com importancias zeradas para termos um modelo final mais parcimonioso.

### Justificativa dos hiperparâmetros

O conjunto de treino possui 439 observações e 20 variáveis preditoras. Por isso, foi usado um espaço de busca deliberadamente restrito: com poucos dados, modelos muito complexos podem memorizar a amostra e produzir decisões instáveis quando aplicados a novos clientes.

- `learning_rate` entre `0.01` e `0.15`: valores baixos tornam o aprendizado mais gradual e reduzem oscilações; o teto evita atualizações agressivas que poderiam superajustar a base pequena. Do ponto de vista de negócio, isso favorece uma priorização mais consistente de clientes para ações de cobrança e renegociação.
- `num_boost_round` entre `50` e `250`: limita a quantidade total de árvores. O limite inferior permite capturar relações não lineares, enquanto o superior evita custo e complexidade desnecessários. O early stopping interrompe o treinamento quando não há ganho na validação.
- `num_leaves` entre `4` e `24` e `max_depth` entre `2` e `6`: controlam diretamente a complexidade das árvores. Faixas pequenas são adequadas ao volume de dados e reduzem o risco de regras muito específicas para poucos clientes, mantendo capacidade para interações entre perfil, atraso e histórico de contato.
- `min_data_in_leaf` entre `10` e `50`: exige suporte mínimo para cada folha. Isso evita que o modelo crie decisões baseadas em grupos muito pequenos, importante para não direcionar ofertas ou contatos com base em evidência frágil.
- `bagging_fraction` e `feature_fraction` entre `0.7` e `1.0`: introduzem amostragem de linhas e variáveis. A regularização reduz correlação entre árvores e melhora a robustez; manter o limite superior em `1.0` permite que o modelo use toda a informação quando isso for validado pelos dados.
- `lambda_l1` e `lambda_l2` entre `0.0` e `2.0`: regularizam pesos e ajudam a reduzir a influência de combinações instáveis. Isso é útil quando o custo de uma priorização errada inclui esforço operacional, contato indevido ou concessão inadequada de oferta.
- `objective='binary'`: corresponde ao target binário `regularizou_30d`, isto é, a previsão de regularização no horizonte de 30 dias.
- `3` folds estratificados e métrica ROC AUC: preservam a proporção das classes em cada validação e avaliam a capacidade de ordenar clientes por probabilidade, que é mais alinhada à priorização de uma carteira do que à acurácia em um ponto de corte fixo.
- `n_trials=10`: limita o custo computacional e o risco de otimizar excessivamente uma validação pequena. É uma busca inicial suficiente para comparar configurações plausíveis; uma etapa posterior pode ampliar a busca após validação OOT.
- `seed=42`, `feature_fraction_seed=42` e `bagging_seed=42`: garantem reprodutibilidade. `num_threads=1` e `force_col_wise=True` favorecem execução estável em uma base pequena e evitam variações desnecessárias no ambiente.

A variável `score_cobranca_atualizado_30d` e a `flag_treino` já foram excluídas antes do ajuste. A primeira poderia carregar informação posterior ao momento da previsão, gerando data leakage; a segunda identifica a origem da amostra e não representa uma característica disponível para decisão sobre o cliente.

In [5]:
import optuna
import lightgbm as lgb

X_treino = base_treino[features].copy()
for coluna in X_treino.select_dtypes(include=['category', 'object']).columns:
    if pd.api.types.is_categorical_dtype(X_treino[coluna]):
        X_treino[coluna] = X_treino[coluna].cat.codes
    else:
        X_treino[coluna] = pd.factorize(X_treino[coluna])[0]
X_treino = np.ascontiguousarray(X_treino.to_numpy(dtype=np.float32))
y_treino = np.ascontiguousarray(base_treino[target].to_numpy(dtype=np.float32))


def roc_auc_manual(valores_reais, scores):
    ordem = np.argsort(scores)
    ranks = np.empty_like(ordem, dtype=float)
    ranks[ordem] = np.arange(1, len(scores) + 1)
    positivos = valores_reais == 1
    negativos = ~positivos
    quantidade_positivos = positivos.sum()
    quantidade_negativos = negativos.sum()
    soma_ranks_positivos = ranks[positivos].sum()
    return (
        soma_ranks_positivos
        - quantidade_positivos * (quantidade_positivos + 1) / 2
    ) / (quantidade_positivos * quantidade_negativos)


def ks_manual(valores_reais, scores):
    ordem = np.argsort(-scores)
    positivos = (valores_reais[ordem] == 1).astype(float)
    negativos = (valores_reais[ordem] == 0).astype(float)
    distribuicao_positivos = np.cumsum(positivos) / positivos.sum()
    distribuicao_negativos = np.cumsum(negativos) / negativos.sum()
    return np.max(np.abs(distribuicao_positivos - distribuicao_negativos))


def folds_estratificados(valores_reais, quantidade_folds=3, seed=42):
    gerador = np.random.default_rng(seed)
    indices_positivos = gerador.permutation(np.flatnonzero(valores_reais == 1))
    indices_negativos = gerador.permutation(np.flatnonzero(valores_reais == 0))
    folds_positivos = np.array_split(indices_positivos, quantidade_folds)
    folds_negativos = np.array_split(indices_negativos, quantidade_folds)

    for indice_fold in range(quantidade_folds):
        indices_validacao = np.concatenate(
            [folds_positivos[indice_fold], folds_negativos[indice_fold]]
        )
        indices_treino = np.setdiff1d(
            np.arange(len(valores_reais)), indices_validacao
        )
        yield indices_treino, indices_validacao


def objetivo(trial):
    parametros = {
        'objective': 'binary',
        'verbosity': -1,
        'seed': 42,
        'feature_fraction_seed': 42,
        'bagging_seed': 42,
        'num_threads': 1,
        'force_col_wise': True,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 4, 24),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 50),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 2.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 2.0),
    }
    quantidade_boosting = trial.suggest_int('num_boost_round', 50, 250)
    resultados = []

    for indices_treino, indices_validacao in folds_estratificados(y_treino):
        dados_treino = lgb.Dataset(
            X_treino[indices_treino],
            label=y_treino[indices_treino],
            feature_name=features,
        )
        dados_validacao = lgb.Dataset(
            X_treino[indices_validacao],
            label=y_treino[indices_validacao],
            feature_name=features,
            reference=dados_treino,
        )
        modelo = lgb.train(
            parametros,
            dados_treino,
            num_boost_round=quantidade_boosting,
            valid_sets=[dados_validacao],
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
        previsoes = modelo.predict(X_treino[indices_validacao])
        resultados.append(roc_auc_manual(y_treino[indices_validacao], previsoes))

    return np.mean(resultados)


estudo_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
estudo_lgbm.optimize(objetivo, n_trials=10)

melhores_parametros = estudo_lgbm.best_params.copy()
quantidade_boosting = melhores_parametros.pop('num_boost_round')
melhores_parametros.update({
    'num_threads': 1,
    'force_col_wise': True,
})
melhor_modelo = lgb.train(
    melhores_parametros,
    lgb.Dataset(X_treino, label=y_treino, feature_name=features),
    num_boost_round=quantidade_boosting,
)

importancias = (
    pd.DataFrame({
        'feature': features,
        'importancia': melhor_modelo.feature_importance(importance_type='gain'),
    })
    .sort_values('importancia', ascending=False)
    .reset_index(drop=True)
)

[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 20
[LightGBM] [Info] Start training from score 0.512528
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

Com a lista de importâncias abaixo, podemos visualizar quais variáveis possuem importância zerada ou muito pequena. Estas serão removidas da modelagem final.

In [6]:
print(f'Melhor ROC AUC médio: {estudo_lgbm.best_value:.4f}')
print('Importância das variáveis:')
print(importancias.to_string(index=False))
features = list(set(features)-set(importancias.loc[importancias['importancia'] <= 2, 'feature'].tolist()))
len(features)

Melhor ROC AUC médio: 0.6963
Importância das variáveis:
                     feature  importancia
  score_regularizacao_legado    91.581771
                 dias_atraso    84.098640
       renegociacoes_ult_12m    24.822768
 faturamento_mensal_estimado    20.207803
  tempo_relacionamento_meses    18.876899
       utilizacao_limite_pct    18.008628
                      regiao    12.814409
              limite_credito    11.630232
      canal_preferencial_cat    10.826896
 dias_desde_ultima_interacao     9.828720
               saldo_devedor     6.941929
      qtd_interacoes_ate_ref     3.193211
  promessa_pagamento_ult_30d     3.015801
qtd_canais_distintos_ate_ref     2.899868
          risco_setorial_cat     2.757019
                   porte_cat     2.569316
        qtd_contratos_ativos     0.893991
        qtd_contatos_ult_30d     0.451481
       qtd_parcelas_vencidas     0.000000
                   setor_cat     0.000000


16

Abaixo, seguiremos para a modelagem final.

In [7]:
optuna.logging.set_verbosity(optuna.logging.CRITICAL)

bases_avaliacao = pd.concat(
    [base_treino[features], base_teste[features], base_oot[features]],
    ignore_index=True,
).copy()

for coluna in bases_avaliacao.select_dtypes(include=['category', 'object']).columns:
    if isinstance(bases_avaliacao[coluna].dtype, pd.CategoricalDtype):
        bases_avaliacao[coluna] = bases_avaliacao[coluna].cat.codes
    else:
        bases_avaliacao[coluna] = pd.factorize(bases_avaliacao[coluna])[0]

X_treino_final = np.ascontiguousarray(
    bases_avaliacao.iloc[:len(base_treino)].to_numpy(dtype=np.float32)
)
X_teste_final = np.ascontiguousarray(
    bases_avaliacao.iloc[len(base_treino):len(base_treino) + len(base_teste)].to_numpy(
        dtype=np.float32
    )
)
X_oot_final = np.ascontiguousarray(
    bases_avaliacao.iloc[len(base_treino) + len(base_teste):].to_numpy(dtype=np.float32)
)
y_treino_final = np.ascontiguousarray(
    base_treino[target].to_numpy(dtype=np.float32)
)
y_teste_final = np.ascontiguousarray(
    base_teste[target].to_numpy(dtype=np.float32)
)
y_oot_final = np.ascontiguousarray(
    base_oot[target].to_numpy(dtype=np.float32)
)

X_treino = X_treino_final
y_treino = y_treino_final

estudo_final = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
estudo_final.optimize(objetivo, n_trials=1500, show_progress_bar=True)

parametros_finais = estudo_final.best_params.copy()
quantidade_boosting_final = parametros_finais.pop('num_boost_round')
parametros_finais.update({
    'objective': 'binary',
    'verbosity': -1,
    'seed': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'num_threads': 1,
    'force_col_wise': True,
})
modelo_final = lgb.train(
    parametros_finais,
    lgb.Dataset(X_treino_final, label=y_treino_final, feature_name=features),
    num_boost_round=quantidade_boosting_final,
)

previsoes_teste = modelo_final.predict(X_teste_final)
previsoes_oot = modelo_final.predict(X_oot_final)
importancias_finais = (
    pd.DataFrame({
        'feature': features,
        'importancia': modelo_final.feature_importance(importance_type='gain'),
    })
    .sort_values('importancia', ascending=False)
    .reset_index(drop=True)
)

print('Importância das variáveis do modelo final:')
print(importancias_finais.to_string(index=False))
print(f'\nBase teste - AUC: {roc_auc_manual(y_teste_final, previsoes_teste):.4f}')
print(f'Base teste - KS: {ks_manual(y_teste_final, previsoes_teste):.4f}')
print(f'Base OOT - AUC: {roc_auc_manual(y_oot_final, previsoes_oot):.4f}')
print(f'Base OOT - KS: {ks_manual(y_oot_final, previsoes_oot):.4f}')

Best trial: 306. Best value: 0.716452: 100%|██████████| 1500/1500 [02:48<00:00,  8.91it/s]


Importância das variáveis do modelo final:
                     feature  importancia
  score_regularizacao_legado   264.530311
                 dias_atraso   249.706919
 faturamento_mensal_estimado   131.909652
  tempo_relacionamento_meses   100.668639
              limite_credito    75.870309
 dias_desde_ultima_interacao    70.229106
       utilizacao_limite_pct    69.256678
               saldo_devedor    67.153256
                      regiao    56.741931
       renegociacoes_ult_12m    53.978863
      canal_preferencial_cat    44.470056
      qtd_interacoes_ate_ref    27.227627
          risco_setorial_cat    23.075562
qtd_canais_distintos_ate_ref    19.535923
                   porte_cat    16.504125
  promessa_pagamento_ult_30d    10.812169

Base teste - AUC: 0.7002
Base teste - KS: 0.3434
Base OOT - AUC: 0.6364
Base OOT - KS: 0.2481


In [8]:
import pickle

pasta_modelos = Path('modelos')
pasta_modelos.mkdir(parents=True, exist_ok=True)

caminho_modelo = pasta_modelos / 'modelo_unico.pkl'

with caminho_modelo.open('wb') as arquivo:
    pickle.dump(modelo_final, arquivo)

print(f'Modelo salvo em: {caminho_modelo}')

Modelo salvo em: modelos\modelo_unico.pkl
